# Indian Sign Language Training Notebook

This Colab notebook downloads or ingests an Indian Sign Language dataset, extracts MediaPipe Holistic landmarks, trains a sequence model, and exports app-compatible `model.tflite`, `labels.json`, and `training_config.json`.

Recommended default: Kaggle `prasadshet/indian-sign-language-video-dataset`, because it is word-level video data. Other useful sources are Kaggle `drblack00/isl-csltr-indian-sign-language-dataset` for sentence-level ISL-CSLTR work, Hugging Face `silentone0725/Indian_Sign_Language_Data.gov_Rencoded` for a much larger ISLRTC/Data.gov dictionary collection, and Kaggle image datasets such as `prathumarikeri/indian-sign-language-isl` for alphabet classifiers.


In [ ]:
!pip -q install mediapipe opencv-python-headless kaggle tqdm scikit-learn pyarrow huggingface_hub


In [ ]:
from pathlib import Path
import hashlib, json, os, re, shutil, subprocess, zipfile
import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

try:
    from google.colab import files, drive
    IN_COLAB = True
except Exception:
    IN_COLAB = False

print('TensorFlow:', tf.__version__)
print('MediaPipe:', mp.__version__)


## Configuration

The notebook defaults to a 10-word experiment so extraction and training are manageable in Colab. Edit `SELECTED_WORDS` after the manifest preview if you want a different set; use `SELECTED_WORDS = None` and `MAX_CLASSES = None` only when you are ready for the full dataset.


In [ ]:
DATASET_SOURCE = 'kaggle'  # kaggle, huggingface, drive, zip_upload
KAGGLE_DATASET = 'prasadshet/indian-sign-language-video-dataset'
HF_DATASET = 'silentone0725/Indian_Sign_Language_Data.gov_Rencoded'
DRIVE_DATASET_DIR = '/content/drive/MyDrive/isl_dataset'

WORK_DIR = Path('/content/signsense_isl')
DATA_ROOT = WORK_DIR / 'data'
CACHE_DIR = WORK_DIR / 'landmark_cache'
OUTPUT_DIR = WORK_DIR / 'artifacts'

MAX_FRAMES = 64
FRAME_STRIDE = 2
SELECTED_WORDS = [
    'Hello',
    'Thank you',
    'Good Morning',
    'Good afternoon',
    'Key',
    'Knife',
    'Switch',
    'Pour',
    'Clean',
    'Maybe',
]
MAX_CLASSES = 10
MAX_SAMPLES_PER_CLASS = 40
MIN_SAMPLES_PER_CLASS = 2
TEST_SIZE = 0.2
RANDOM_SEED = 42
EPOCHS = 40
BATCH_SIZE = 32
LEARNING_RATE = 5e-4

for path in [WORK_DIR, DATA_ROOT, CACHE_DIR, OUTPUT_DIR]:
    path.mkdir(parents=True, exist_ok=True)


In [ ]:
def setup_kaggle_credentials():
    kaggle_json = Path('/root/.kaggle/kaggle.json')
    if kaggle_json.exists():
        return
    if not IN_COLAB:
        raise RuntimeError('Place kaggle.json at ~/.kaggle/kaggle.json or run this notebook in Colab.')
    print('Upload kaggle.json from Kaggle > Account > API > Create New Token.')
    uploaded = files.upload()
    if 'kaggle.json' not in uploaded:
        raise RuntimeError('kaggle.json was not uploaded.')
    kaggle_json.parent.mkdir(parents=True, exist_ok=True)
    Path('kaggle.json').replace(kaggle_json)
    os.chmod(kaggle_json, 0o600)


def download_dataset():
    if DATASET_SOURCE == 'kaggle':
        setup_kaggle_credentials()
        subprocess.run(['kaggle', 'datasets', 'download', '-d', KAGGLE_DATASET, '-p', str(DATA_ROOT), '--unzip'], check=True)
        return DATA_ROOT
    if DATASET_SOURCE == 'huggingface':
        from huggingface_hub import snapshot_download
        snapshot_download(repo_id=HF_DATASET, repo_type='dataset', local_dir=str(DATA_ROOT))
        return DATA_ROOT
    if DATASET_SOURCE == 'drive':
        if IN_COLAB:
            drive.mount('/content/drive')
        source = Path(DRIVE_DATASET_DIR)
        if not source.exists():
            raise FileNotFoundError(source)
        return source
    if DATASET_SOURCE == 'zip_upload':
        if not IN_COLAB:
            raise RuntimeError('zip_upload is intended for Colab.')
        uploaded = files.upload()
        zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
        if not zip_names:
            raise RuntimeError('Upload a .zip dataset archive.')
        with zipfile.ZipFile(zip_names[0], 'r') as zf:
            zf.extractall(DATA_ROOT)
        return DATA_ROOT
    raise ValueError(f'Unknown DATASET_SOURCE: {DATASET_SOURCE}')


DATASET_DIR = download_dataset()
print('Dataset directory:', DATASET_DIR)


In [ ]:
VIDEO_EXTS = {'.mp4', '.avi', '.mov', '.mkv', '.webm'}
LANDMARK_EXTS = {'.parquet', '.pq'}
GENERIC_DIR_NAMES = {'video', 'videos', 'data', 'dataset', 'train', 'test', 'val', 'valid', 'validation'}


def clean_label(label):
    label = str(label).replace('_', ' ').replace('-', ' ')
    return re.sub(r'\s+', ' ', label).strip()


def infer_label(path, root):
    rel = path.relative_to(root)
    parent = rel.parts[-2] if len(rel.parts) > 1 else ''
    if parent and parent.lower() not in GENERIC_DIR_NAMES:
        return clean_label(parent)
    stem = re.sub(r'\d+$', '', path.stem)
    stem = re.split(r'[_\- ](?:sample|video|vid|clip)?\d+', stem, maxsplit=1)[0]
    return clean_label(stem or path.stem)


def cap_samples_per_class(frame):
    if MAX_SAMPLES_PER_CLASS is None:
        return frame
    return frame.groupby('label', group_keys=False).apply(
        lambda group: group.sample(n=min(MAX_SAMPLES_PER_CLASS, len(group)), random_state=RANDOM_SEED)
    )


def normalize_label_for_match(label):
    return clean_label(label).casefold()


def choose_labels(manifest):
    counts = manifest['label'].value_counts()
    valid_counts = counts[counts >= MIN_SAMPLES_PER_CLASS]
    if SELECTED_WORDS:
        requested = {normalize_label_for_match(label): label for label in SELECTED_WORDS}
        available = {normalize_label_for_match(label): label for label in valid_counts.index}
        matched = [available[key] for key in requested if key in available]
        missing = [label for key, label in requested.items() if key not in available]
        if missing:
            print('Requested words not found in this dataset:', missing)
            print('Available labels preview:')
            display(valid_counts.head(40))
        if len(matched) < 2:
            raise RuntimeError('Fewer than 2 requested words were found. Edit SELECTED_WORDS to match labels shown above.')
        return matched
    if MAX_CLASSES is not None:
        return valid_counts.head(MAX_CLASSES).index.tolist()
    return valid_counts.index.tolist()


def build_manifest(root):
    root = Path(root)
    found = [p for p in root.rglob('*') if p.suffix.lower() in VIDEO_EXTS | LANDMARK_EXTS]
    manifest = pd.DataFrame([{'path': str(p), 'label': infer_label(p, root)} for p in found])
    if manifest.empty:
        raise RuntimeError(f'No videos or parquet landmark files found under {root}')
    keep = choose_labels(manifest)
    manifest = manifest[manifest['label'].isin(keep)]
    manifest = cap_samples_per_class(manifest)
    return manifest.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)


manifest = build_manifest(DATASET_DIR)
manifest.to_csv(WORK_DIR / 'manifest.csv', index=False)
print('Samples/classes:', manifest.shape[0], manifest['label'].nunique())
display(manifest.head())
display(manifest['label'].value_counts())


In [ ]:
LANDMARK_SPECS = (('face', 468), ('pose', 33), ('left_hand', 21), ('right_hand', 21))
DIM_NAMES = ('x', 'y', 'z')
ROWS_PER_FRAME = sum(count for _, count in LANDMARK_SPECS)
MODEL_URL = 'https://storage.googleapis.com/mediapipe-models/holistic_landmarker/holistic_landmarker/float16/1/holistic_landmarker.task'
MODEL_PATH = WORK_DIR / 'holistic_landmarker.task'


def build_landmark_template():
    return pd.DataFrame([{'type': t, 'landmark_index': i} for t, n in LANDMARK_SPECS for i in range(n)])


XYZ_SKEL = build_landmark_template()


def ensure_model_asset():
    if MODEL_PATH.exists():
        return MODEL_PATH
    import urllib.request
    print('Downloading MediaPipe Holistic Landmarker task bundle...')
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
    return MODEL_PATH


def unwrap_landmark_group(group):
    if group is None:
        return None
    if isinstance(group, (list, tuple)):
        if not group:
            return None
        first = group[0]
        if hasattr(first, 'landmark'):
            return first.landmark
        if isinstance(first, (list, tuple)):
            return first
        return group
    if hasattr(group, 'landmark'):
        return group.landmark
    return None


def landmarks_to_dataframe(landmarks, type_name):
    landmarks = unwrap_landmark_group(landmarks)
    if not landmarks:
        return pd.DataFrame(columns=['landmark_index', 'x', 'y', 'z', 'type'])
    return pd.DataFrame([
        {'landmark_index': i, 'x': getattr(p, 'x', np.nan), 'y': getattr(p, 'y', np.nan), 'z': getattr(p, 'z', np.nan), 'type': type_name}
        for i, p in enumerate(landmarks)
    ])


def results_to_frame_array(result):
    parts = []
    for t, _ in LANDMARK_SPECS:
        group_df = landmarks_to_dataframe(getattr(result, f'{t}_landmarks', None), t)
        if not group_df.empty:
            parts.append(group_df)
    landmarks = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=['landmark_index', 'x', 'y', 'z', 'type'])
    frame_df = XYZ_SKEL.merge(landmarks, on=['type', 'landmark_index'], how='left')
    return frame_df[['x', 'y', 'z']].to_numpy(dtype=np.float32)


def pad_or_trim(sequence):
    sequence = np.nan_to_num(sequence.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)
    if sequence.shape[0] >= MAX_FRAMES:
        return sequence[-MAX_FRAMES:]
    padded = np.zeros((MAX_FRAMES, ROWS_PER_FRAME, len(DIM_NAMES)), dtype=np.float32)
    padded[-sequence.shape[0]:] = sequence
    return padded



FEATURE_IDXS = np.array(
    list(range(468, 489)) +
    list(range(489, 522)) +
    list(range(522, 543)) +
    [61, 185, 40, 39, 37, 0, 267, 269, 270, 409, 291, 146, 91, 181, 84, 17, 314, 405, 321, 375],
    dtype=np.int32,
)
N_FEATURE_POINTS = len(FEATURE_IDXS)


def select_features(sequence):
    return sequence[:, FEATURE_IDXS, :]

ensure_model_asset()
print('Rows per frame:', ROWS_PER_FRAME, 'Feature points:', N_FEATURE_POINTS)


In [ ]:
def cache_path_for(path):
    digest = hashlib.sha1(str(path).encode('utf-8')).hexdigest()[:16]
    return CACHE_DIR / f'{Path(path).stem}-{digest}.npy'


def load_parquet_sequence(path):
    data = pd.read_parquet(path, columns=list(DIM_NAMES)).to_numpy(dtype=np.float32)
    if len(data) % ROWS_PER_FRAME != 0:
        raise ValueError(f'{path} has {len(data)} rows, not divisible by {ROWS_PER_FRAME}')
    return data.reshape(-1, ROWS_PER_FRAME, len(DIM_NAMES))


def extract_video_sequence(path, detector, start_timestamp_ms=0):
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        raise RuntimeError(f'Could not open video: {path}')
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    frame_idx = 0
    last_timestamp_ms = start_timestamp_ms
    target_size = None
    frames = []
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if FRAME_STRIDE > 1 and frame_idx % FRAME_STRIDE != 0:
            frame_idx += 1
            continue
        if target_size is None:
            target_size = (frame.shape[1], frame.shape[0])
        elif frame.shape[1] != target_size[0] or frame.shape[0] != target_size[1]:
            frame = cv2.resize(frame, target_size, interpolation=cv2.INTER_AREA)
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        timestamp_ms = max(last_timestamp_ms + 1, start_timestamp_ms + int((frame_idx / fps) * 1000))
        last_timestamp_ms = timestamp_ms
        result = detector.detect_for_video(mp_image, timestamp_ms)
        frames.append(results_to_frame_array(result))
        frame_idx += 1
    cap.release()
    if not frames:
        return np.zeros((0, ROWS_PER_FRAME, len(DIM_NAMES)), dtype=np.float32), last_timestamp_ms + 1
    return np.stack(frames).astype(np.float32), last_timestamp_ms + 1


def load_or_extract_sequence(path, detector, start_timestamp_ms=0):
    path = Path(path)
    cached = cache_path_for(path)
    if cached.exists():
        return np.load(cached), start_timestamp_ms
    if path.suffix.lower() in LANDMARK_EXTS:
        sequence = load_parquet_sequence(path)
        next_timestamp_ms = start_timestamp_ms
    else:
        sequence, next_timestamp_ms = extract_video_sequence(path, detector, start_timestamp_ms=start_timestamp_ms)
    np.save(cached, sequence)
    return sequence, next_timestamp_ms


base_options = python.BaseOptions(model_asset_path=str(MODEL_PATH))
options = vision.HolisticLandmarkerOptions(base_options=base_options, running_mode=vision.RunningMode.VIDEO)

X, kept_labels, failed = [], [], []
next_timestamp_ms = 0

with vision.HolisticLandmarker.create_from_options(options) as detector:
    for row in tqdm(manifest.itertuples(index=False), total=len(manifest)):
        try:
            seq, next_timestamp_ms = load_or_extract_sequence(row.path, detector, start_timestamp_ms=next_timestamp_ms)
            if seq.shape[0] == 0:
                failed.append({'path': row.path, 'label': row.label, 'error': 'empty sequence'})
                continue
            X.append(select_features(pad_or_trim(seq)))
            kept_labels.append(row.label)
        except Exception as exc:
            failed.append({'path': row.path, 'label': row.label, 'error': str(exc)})
            next_timestamp_ms += 1

if failed:
    failed_path = WORK_DIR / 'failed_extractions.csv'
    pd.DataFrame(failed).to_csv(failed_path, index=False)
    print(f'Saved extraction failures to {failed_path}')

if not X:
    raise RuntimeError('No usable samples were extracted. Check failed_extractions.csv and verify your dataset paths contain readable videos or parquet landmark files.')

labels = sorted(set(kept_labels))
label_to_id = {label: i for i, label in enumerate(labels)}
X = np.stack(X).astype(np.float32)
y = np.asarray([label_to_id[label] for label in kept_labels], dtype=np.int64)

print('X:', X.shape, 'y:', y.shape, 'classes:', len(labels), 'failed:', len(failed))
display(pd.Series(kept_labels, name='label').value_counts().head(30))
if len(X) < 2:
    raise RuntimeError('Only one usable sample was extracted. Add more videos, loosen MAX_CLASSES/MAX_SAMPLES_PER_CLASS, or inspect failed_extractions.csv before training.')


In [ ]:
def make_train_val_split(y):
    n_samples = len(y)
    class_counts = pd.Series(y).value_counts()
    n_classes = len(class_counts)

    if n_samples < 2:
        raise RuntimeError('Need at least 2 usable samples to create a train/validation split.')

    n_val = max(1, int(np.ceil(TEST_SIZE * n_samples)))
    if n_samples - n_val < 1:
        n_val = 1

    can_stratify = (
        n_classes > 1
        and class_counts.min() >= 2
        and n_val >= n_classes
        and (n_samples - n_val) >= n_classes
    )
    stratify_labels = y if can_stratify else None

    if not can_stratify:
        print('Using non-stratified split because the extracted dataset is small or has classes with only one sample.')
        print('Class counts:')
        display(class_counts.rename(index=lambda idx: labels[int(idx)]).head(30))

    return train_test_split(
        np.arange(n_samples),
        test_size=n_val,
        random_state=RANDOM_SEED,
        stratify=stratify_labels,
    )


train_idx, val_idx = make_train_val_split(y)
X_train, X_val = X[train_idx], X[val_idx]
y_train, y_val = y[train_idx], y[val_idx]
print('Train:', X_train.shape, 'Validation:', X_val.shape)


In [ ]:
def augment_sequence(batch):
    noise = tf.random.normal(tf.shape(batch), stddev=0.01)
    scale = tf.random.uniform([tf.shape(batch)[0], 1, 1, 1], 0.9, 1.1)
    return batch * scale + noise


def build_model(num_classes):
    inputs = tf.keras.layers.Input(shape=(MAX_FRAMES, N_FEATURE_POINTS, len(DIM_NAMES)), name='inputs')
    x = tf.keras.layers.Lambda(lambda t: tf.where(tf.math.is_nan(t), tf.zeros_like(t), t))(inputs)
    x = tf.keras.layers.Lambda(lambda t: augment_sequence(t))(x)
    x = tf.keras.layers.Reshape((MAX_FRAMES, N_FEATURE_POINTS * len(DIM_NAMES)))(x)
    x = tf.keras.layers.LayerNormalization()(x)
    x = tf.keras.layers.Masking(mask_value=0.0)(x)
    x = tf.keras.layers.Dense(128, activation='gelu')(x)
    x = tf.keras.layers.Bidirectional(tf.keras.layers.GRU(96, return_sequences=True))(x)
    x = tf.keras.layers.Bidirectional(tf.keras.layers.GRU(64))(x)
    x = tf.keras.layers.Dense(128, activation='gelu')(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    outputs = tf.keras.layers.Dense(num_classes, activation='softmax', name='outputs')(x)
    model = tf.keras.Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer=tf.keras.optimizers.Adam(LEARNING_RATE), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model


model = build_model(len(labels))
model.summary()


In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=6, restore_best_weights=True, monitor='val_accuracy', mode='max'),
    tf.keras.callbacks.ReduceLROnPlateau(patience=3, factor=0.5, monitor='val_loss'),
    tf.keras.callbacks.ModelCheckpoint(str(OUTPUT_DIR / 'best_model.keras'), save_best_only=True, monitor='val_accuracy', mode='max'),
]

history = model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=EPOCHS, batch_size=BATCH_SIZE, callbacks=callbacks)


In [ ]:
val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)
print({'val_loss': float(val_loss), 'val_accuracy': float(val_acc)})
pred = model.predict(X_val[:10], verbose=0).argmax(axis=1)
for expected, got in zip(y_val[:10], pred):
    print('expected:', labels[int(expected)], '| predicted:', labels[int(got)])


In [ ]:
model.save(OUTPUT_DIR / 'keras_model.keras')
(OUTPUT_DIR / 'labels.json').write_text(json.dumps({'labels': labels, 'label_to_id': label_to_id}, indent=2), encoding='utf-8')
(OUTPUT_DIR / 'training_config.json').write_text(json.dumps({
    'max_frames': MAX_FRAMES,
    'frame_stride': FRAME_STRIDE,
    'rows_per_frame': ROWS_PER_FRAME,
    'feature_indices': FEATURE_IDXS.tolist(),
    'n_feature_points': N_FEATURE_POINTS,
    'dims': list(DIM_NAMES),
    'dataset_source': DATASET_SOURCE,
    'kaggle_dataset': KAGGLE_DATASET if DATASET_SOURCE == 'kaggle' else None,
    'num_classes': len(labels),
    'history': {k: [float(v) for v in vals] for k, vals in history.history.items()},
}, indent=2), encoding='utf-8')


class InferenceModule(tf.Module):
    def __init__(self, keras_model):
        super().__init__()
        self.keras_model = keras_model

    @tf.function(input_signature=[tf.TensorSpec(shape=[MAX_FRAMES, N_FEATURE_POINTS, len(DIM_NAMES)], dtype=tf.float32, name='inputs')])
    def __call__(self, inputs):
        outputs = self.keras_model(tf.expand_dims(inputs, axis=0), training=False)[0]
        return {'outputs': outputs}


module = InferenceModule(model)
converter = tf.lite.TFLiteConverter.from_concrete_functions([module.__call__.get_concrete_function()], module)
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS, tf.lite.OpsSet.SELECT_TF_OPS]
converter._experimental_lower_tensor_list_ops = False
converter.optimizations = [tf.lite.Optimize.DEFAULT]
(OUTPUT_DIR / 'model.tflite').write_bytes(converter.convert())

print('Artifacts:')
for path in sorted(OUTPUT_DIR.iterdir()):
    print(path.name, path.stat().st_size)


In [ ]:
interpreter = tf.lite.Interpreter(model_path=str(OUTPUT_DIR / 'model.tflite'))
interpreter.allocate_tensors()
pred_fn = interpreter.get_signature_runner('serving_default')
result = pred_fn(inputs=X_val[0])
print('TFLite predicted:', labels[int(result['outputs'].argmax())])
print('Expected:', labels[int(y_val[0])])


In [ ]:
archive_path = shutil.make_archive('/content/signsense_isl_artifacts', 'zip', OUTPUT_DIR)
print('Created:', archive_path)
if IN_COLAB:
    files.download(archive_path)
